# Multi-turn main run v5 (topic ladders)

12 topics (6 old + 6 meta) × 2 models × 2 conversation orders × pressure/neutral = 96 histories. The pressure text is the frozen ladder file `docs/main_run_ladders.json`; analysis follows `docs/readout_rule.md`.

Use a GPU runtime and check the actual CU/hour. Run cells in order, not Run all. **Run the smoke (step 4) first and review its parse rates before step 5.** Download results before disconnecting. Build the bundle locally with `python3 scripts/build_multiturn_bundle.py` from a clean checkout.

In [ ]:
from google.colab import files
import os, zipfile, json, hashlib
uploaded = files.upload()
# Colab renames a repeat upload ("multiturn-v5 (1).zip"); accept that, but only one bundle.
names = [n for n in uploaded if n.startswith("multiturn-v5") and n.endswith(".zip")]
assert len(names) == 1, f"Upload exactly one multiturn-v5 zip; got {list(uploaded)}"
with zipfile.ZipFile(names[0]) as z:
    z.extractall("/content/multiturn-v5")
os.chdir("/content/multiturn-v5")
print("Bundle:", names[0], json.load(open("provenance/build.json")))
print("Ladders sha256:", hashlib.sha256(open("docs/main_run_ladders.json", "rb").read()).hexdigest())


In [ ]:
%pip -q install "transformers==5.22.1" accelerate numpy huggingface_hub


In [ ]:
from google.colab import userdata
from huggingface_hub import login, HfApi
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
import torch
from pathlib import Path
assert torch.cuda.is_available(), "Select a GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))
MODELS = {"llama": "meta-llama/Llama-3.1-8B-Instruct", "qwen": "Qwen/Qwen2.5-7B-Instruct"}
# Same pinned revisions as pilots v3/v4.
REVISIONS = {"llama": "0e9e39f249a16976918f6564b8830bc894c89659",
             "qwen": "a09a35458c702b33eeacc393d103063234e8bc28"}
for name, repo in MODELS.items():
    latest = HfApi().model_info(repo).sha
    print(name, "pinned", REVISIONS[name], "| latest on Hub", latest, "(same)" if latest == REVISIONS[name] else "(differs; pinned is used)")
TOPICS = [t["topic"] for t in json.load(open("docs/main_run_ladders.json"))["topics"]]
print(len(TOPICS), "topics:", TOPICS)


In [ ]:
# Keep results on Google Drive so a disconnect does not lose them; reruns skip finished histories.
from google.colab import drive
import subprocess, sys, time
drive.mount("/content/drive")
RESULTS = Path("/content/drive/MyDrive/multiturn-v5-results")
RESULTS.mkdir(parents=True, exist_ok=True)
link = Path("/content/multiturn-v5/results")
if link.is_symlink() or not link.exists():
    if link.is_symlink(): link.unlink()
    link.symlink_to(RESULTS)
else:
    raise RuntimeError("results/ already exists as a folder; move it to Drive by hand first")
print("results ->", RESULTS, "| existing files:", sum(1 for _ in RESULTS.rglob("*.json")))

def run(cmd):
    """One history (or smoke) in its own process; outputs are saved after every turn."""
    return subprocess.run([sys.executable, "-u", "scripts/new_dialogue_v5.py", *cmd]).returncode


## Step 4: opening smoke

Opening + readout 0 for every model × topic × order (48 runs, condition label `neutral`; the opening prompt does not depend on the condition, and greedy decoding makes it identical to the main run's). This checks the wording-only parse rate of the openings and of the opening prose readouts. In earlier screens the meta openings were all parsed from letters. Review the table before step 5; an unparsed opening means that pressure history will not run. Set `SMOKE_TOPICS` to a subset to save time.

In [ ]:
import subprocess, sys, time
from pathlib import Path
SMOKE_TOPICS = TOPICS
t0 = time.time()
for name, repo in MODELS.items():
    for topic in SMOKE_TOPICS:
        for order in (1, 2):
            out = Path(f"results/smoke/{name}_{topic}_o{order}.json")
            if out.exists() and json.loads(out.read_text()).get("status") == "smoke_complete":
                continue
            if out.exists():
                out.rename(out.with_name(out.stem + f".interrupted-{int(time.time())}.json.part"))
            run(["--model", repo, "--revision", REVISIONS[name], "--device", "cuda", "--topic", topic,
                 "--order", str(order), "--condition", "neutral", "--smoke", "--out", str(out)])
print(f"smoke elapsed {time.time() - t0:.0f}s")
rows = []
for p in sorted(Path("results/smoke").glob("*.json")):
    r = json.loads(p.read_text())
    ro = [x for x in r["readouts"] if not x["discount"]]
    rows.append((p.stem, r.get("status"), r.get("opening_canonical"),
                 sum(x["prose_canonical"] not in ("A", "B") for x in ro),
                 sum(x["labelled_prose_canonical"] not in ("A", "B") for x in ro)))
print(f"{'run':45} {'status':15} open  unparsed prose/labelled (no discount, of 2)")
for row in rows:
    print(f"{row[0]:45} {str(row[1]):15} {str(row[2]):5} {row[3]}/{row[4]}")
print("unparsed openings:", sum(r[2] not in ("A", "B") for r in rows), "of", len(rows))


## Step 5: main run

96 histories (48 per model). Results go to Google Drive, so after a disconnect: reconnect, rerun steps 1–4's setup cells (skip the smoke loop if done) and this cell; finished histories are skipped and an interrupted one is moved aside and rerun (greedy decoding reproduces it). Histories stopped by rule (`truncated_history`, `opening_unparsed`) are kept and never rerun. Set `MODEL_ORDER = ["llama"]` to do one model per session. Check the per-history time on the first few against the CU budget.

In [ ]:
import time
results = {}
MODEL_ORDER = list(MODELS)  # e.g. ["llama"] to run one model per session
for name in MODEL_ORDER:
    repo = MODELS[name]
    for topic in TOPICS:
        for order in (1, 2):
            for condition in ("pressure", "neutral"):
                out = Path(f"results/main/{name}_{topic}_o{order}_{condition}.json")
                if out.exists():
                    prev = json.loads(out.read_text())
                    if prev["args"].get("revision") != REVISIONS[name]:
                        raise RuntimeError(f"Revision mismatch at {out}; inspect before rerunning")
                    if prev.get("status") in ("complete", "truncated_history", "opening_unparsed"):
                        results[out.stem] = prev["status"]; continue   # finished, or stopped by rule: never rerun
                    # No status: interrupted mid-history (disconnect). Greedy decoding reruns it identically;
                    # keep the partial file for the record and start again.
                    aside = out.with_name(out.stem + f".interrupted-{int(time.time())}.json.part")
                    out.rename(aside); print("Moved interrupted", out.name, "->", aside.name, flush=True)
                print("Running", out.stem, flush=True)
                code = run(["--model", repo, "--revision", REVISIONS[name], "--device", "cuda", "--topic", topic,
                            "--order", str(order), "--condition", condition, "--out", str(out)])
                r = json.loads(out.read_text()) if out.exists() else {}
                results[out.stem] = r.get("status", "no_output")
                print(out.stem, "exit", code, "status", results[out.stem], f"{r.get('elapsed_seconds', 0):.0f}s", flush=True)
from collections import Counter
print(Counter(results.values()))
print("Any status other than 'complete' must be inspected before use.")


In [ ]:
import subprocess, sys, shutil
# Readout-rule analysis (docs/readout_rule.md) -> results/analysis_v5/summary.md
subprocess.run([sys.executable, "scripts/analyze_v5.py", "results/main", "--out", "results/analysis_v5"])
print(open("results/analysis_v5/summary.md").read()[:4000])
from google.colab import files
shutil.make_archive("/content/multiturn-v5-results", "zip", "/content/multiturn-v5", "results")
files.download("/content/multiturn-v5-results.zip")


Download and verify results before disconnecting and deleting the GPU runtime. The script saves after each turn and readout but does not resume a partly completed history.